# Projet 8 OCR — Analyse sociodémographique des étudiants Data

Visualisations issues du mart dbt `mart_profil_sociodemographique`.

**Avant de démarrer :** monter Google Drive et vérifier que le CSV est dans `MonDrive/ColabProjet8/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.rcParams.update({
    'figure.dpi': 150,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'axes.titlesize': 14,
    'axes.labelsize': 11,
})

BLUE  = '#1f6aa5'
CORAL = '#e05a4e'
TEAL  = '#2a9d8f'
GOLD  = '#e9c46a'
GRAY  = '#adb5bd'

In [ ]:
CSV_PATH = '/content/drive/MyDrive/ColabProjet8/mart_profil_sociodemographique.csv'

df = pd.read_csv(CSV_PATH)
df.columns = df.columns.str.lower()

print(f'{len(df)} lignes chargées')
df.head()

In [ ]:
print(df[df.gender == 'Total'].groupby('year')['student_count'].sum())

## Effectifs par région (2022–2025)

In [ ]:
region_totals = (
    df[df.gender == 'Total']
    .groupby('region')['student_count']
    .sum().dropna()
    .sort_values(ascending=False)
)

idf_pct = region_totals.iloc[0] / region_totals.sum() * 100

fig, ax = plt.subplots(figsize=(13, 5))
bars = ax.bar(region_totals.index, region_totals.values, color=BLUE, width=0.7)

for bar, val in zip(bars, region_totals.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 15,
            f'{int(val):,}', ha='center', va='bottom', fontsize=8.5)

ax.set_title(f'Effectifs OCR par région — Parcours Data 2022–2025\n{region_totals.index[0]} : {idf_pct:.1f}% des inscriptions', pad=12)
ax.set_ylabel("Nombre d'étudiants")
ax.tick_params(axis='x', rotation=40)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.tight_layout()
plt.savefig('/content/graph_regions.png', bbox_inches='tight')
plt.show()

## Évolution annuelle des inscriptions

In [ ]:
yearly = (
    df[df.gender == 'Total']
    .groupby('year')['student_count']
    .sum().dropna().sort_index()
)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(yearly.index.astype(int), yearly.values,
        marker='o', linewidth=2.5, markersize=8, color=BLUE)

for year, val in yearly.items():
    ax.annotate(f'{int(val):,}', xy=(year, val), xytext=(0, 12),
                textcoords='offset points', ha='center',
                fontsize=10, fontweight='bold', color=BLUE)

ax.set_title("Évolution des inscriptions au parcours Data OCR", pad=12)
ax.set_ylabel("Nombre d'étudiants")
ax.set_xlabel('Année')
ax.set_xticks(yearly.index.astype(int))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.set_ylim(0, yearly.max() * 1.25)

plt.tight_layout()
plt.savefig('/content/graph_yearly.png', bbox_inches='tight')
plt.show()

## Taux de genre non renseigné par année

In [ ]:
nr = df[df.gender == 'Non renseigne'].groupby('year')['student_count'].sum().dropna()
total_yr = df[df.gender == 'Total'].groupby('year')['student_count'].sum().dropna()
nr_rate = (nr / total_yr * 100).sort_index()

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(nr_rate.index.astype(int), nr_rate.values, color=CORAL, width=0.6)

for bar, val in zip(bars, nr_rate.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_title('Taux de genre non renseigné par année', pad=12)
ax.set_ylabel('% des inscriptions')
ax.set_xlabel('Année')
ax.set_xticks(nr_rate.index.astype(int))
ax.set_ylim(0, nr_rate.max() * 1.25)
ax.axhline(y=10, color=GRAY, linestyle='--', linewidth=1, label='Seuil 10%')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('/content/graph_nr_rate.png', bbox_inches='tight')
plt.show()

## Taux d'inscription pour 100 000 habitants par région

In [ ]:
bench = (
    df[
        (df.gender == 'Total') &
        (df.region != 'Corse') &
        (df.students_per_100k.notna())
    ]
    .groupby('region')['students_per_100k']
    .mean()
    .sort_values(ascending=False)
)

fig, ax = plt.subplots(figsize=(13, 5))
bars = ax.bar(bench.index, bench.values, color=TEAL, width=0.7)

for bar, val in zip(bars, bench.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f'{val:.1f}', ha='center', va='bottom', fontsize=8.5)

mean_val = bench.mean()
ax.axhline(y=mean_val, color=GOLD, linestyle='--', linewidth=1.5,
           label=f'Moyenne : {mean_val:.1f}')
ax.set_title("Étudiants OCR pour 100 000 habitants — moyenne 2022–2025\n(hors Corse : 0 étudiant OCR)", pad=12)
ax.set_ylabel('Étudiants / 100 000 hab.')
ax.tick_params(axis='x', rotation=40)
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('/content/graph_100k.png', bbox_inches='tight')
plt.show()

## Répartition H/F par année (genre renseigné uniquement)

In [ ]:
gender_known = (
    df[df.gender.isin(['M', 'F'])]
    .groupby(['year', 'gender'])['student_count']
    .sum().dropna()
    .unstack(fill_value=0)
    .sort_index()
)
gender_pct = gender_known.div(gender_known.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(8, 4))
x = gender_pct.index.astype(int)
ax.bar(x, gender_pct['M'], label='Hommes', color=BLUE, width=0.5)
ax.bar(x, gender_pct['F'], bottom=gender_pct['M'], label='Femmes', color=CORAL, width=0.5)

for year in x:
    m = gender_pct.loc[year, 'M']
    f = gender_pct.loc[year, 'F']
    ax.text(year, m/2, f'{m:.0f}%', ha='center', va='center',
            color='white', fontsize=10, fontweight='bold')
    ax.text(year, m + f/2, f'{f:.0f}%', ha='center', va='center',
            color='white', fontsize=10, fontweight='bold')

ax.set_title('Répartition H/F par année', pad=12)
ax.set_ylabel('%')
ax.set_xticks(x)
ax.set_ylim(0, 105)
ax.legend(loc='upper right')

plt.tight_layout()
plt.savefig('/content/graph_gender_split.png', bbox_inches='tight')
plt.show()

In [ ]:
import os
for f in sorted(f for f in os.listdir('/content') if f.endswith('.png')):
    print(f'✓ {f}')